### SLMs - Small language models, example 3, using road data (Excel)

**This example uses SmolLM2 1.7B -version. Switch to 360M version if running on CPU instead of GPU (Nvidia GPU required + PyTorch with GPU-support installed)**

In [1]:
# this is the GPU-version of PyTorch, see the website for updated installation commands:
# https://pytorch.org/get-started/locally/

# pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu128

# we also need these:
# pip install transformers datasets peft accelerate trl
# pip install gradio

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from peft import PeftModel

# small configuration tweak to improve inference speed
torch.backends.cuda.matmul.allow_tf32 = True

# let's try the 1.7B or 360M-version of SmolLM2, Instruct-model
# model_name = "HuggingFaceTB/SmolLM2-360M-Instruct"
model_name = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="cuda",
    attn_implementation="sdpa"
)


# switch to GPU if possible
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# small speed boost for inference
model.gradient_checkpointing_disable()

# sanity check, which device is used now
print("\nDevice used: " + device)

c:\SUBASH\LAPIN_AMK\SPRING2026\SML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 218/218 [00:12<00:00, 18.03it/s]



Device used: cuda


In [3]:
import re
import gradio as gr
import io
import contextlib
from datetime import datetime

## Button press data
press_data = {
    1: {"date": "2025-11-10", "time": "01:50", "town": "Turku", "street": "Telekatu", "lat": 60.493152, "lon": 22.288016, "darkness": "It's dark", "acceleration": 44.8, "speed": 0.0, "surface": "Wet", "traffic": "Low", "friction": 0.688},
    2: {"date": "2025-11-10", "time": "04:06", "town": "Salon suuralue", "street": "Meriniitynkatu", "lat": 60.383189, "lon": 23.100664, "darkness": "It's dark", "acceleration": 2083.0, "speed": 37.8, "surface": "Wet", "traffic": "Low", "friction": 0.623},
    3: {"date": "2025-11-10", "time": "04:28", "town": "Salo", "street": "Haarlantie", "lat": 60.204471, "lon": 23.113718, "darkness": "It's dark", "acceleration": 634.6, "speed": 13.8, "surface": "Wet", "traffic": "Low", "friction": 0.746},
    4: {"date": "2025-11-10", "time": "05:40", "town": "Raasepori", "street": "Ekerövägen", "lat": 60.009589, "lon": 23.543561, "darkness": "It's dark", "acceleration": 225.3, "speed": 12.5, "surface": "Wet", "traffic": "Low", "friction": 0.677},
    5: {"date": "2025-11-10", "time": "09:54", "town": "Hanko", "street": "Town Centre", "lat": 59.819316, "lon": 22.958070, "darkness": "It's not dark", "acceleration": 100.2, "speed": 0.4, "surface": "Dry", "traffic": "High", "friction": 0.820},
    6: {"date": "2025-11-10", "time": "18:05", "town": "Salon suuralue", "street": "Otsonkatu", "lat": 60.386477, "lon": 23.159415, "darkness": "It's dark", "acceleration": 108.8, "speed": 20.2, "surface": "Dry", "traffic": "High", "friction": 0.820},
    7: {"date": "2025-11-27", "time": "00:06", "town": "Salo", "street": "Kiskontie", "lat": 60.234862, "lon": 23.588156, "darkness": "It's dark", "acceleration": 352.7, "speed": 2.2, "surface": "Slippery", "traffic": "Low", "friction": 0.464},
    8: {"date": "2025-11-27", "time": "13:06", "town": "Raisio", "street": "Nesteentie", "lat": 60.482399, "lon": 22.165015, "darkness": "It's not dark", "acceleration": 337.5, "speed": 0.9, "surface": "Dry", "traffic": "Medium", "friction": 0.820},
    9: {"date": "2025-11-27", "time": "14:04", "town": "Salo", "street": "Turku–Helsinki moottoritie", "lat": 60.434343, "lon": 22.977084, "darkness": "It's not dark", "acceleration": 209.2, "speed": 93.0, "surface": "Moist", "traffic": "Medium", "friction": 0.810},
    10: {"date": "2025-11-27", "time": "14:57", "town": "Salo", "street": "Kiskontie", "lat": 60.246788, "lon": 23.510027, "darkness": "It's not dark", "acceleration": 109.9, "speed": 79.1, "surface": "Streaming water", "traffic": "Medium", "friction": 0.323},
    11: {"date": "2025-11-27", "time": "15:12", "town": "Tammisaari", "street": "Mustiontie", "lat": 60.181420, "lon": 23.761897, "darkness": "It's not dark", "acceleration": 67.3, "speed": 64.6, "surface": "Streaming water", "traffic": "High", "friction": 0.256},
    12: {"date": "2026-01-09", "time": "01:16", "town": "Raisio", "street": "Nesteentie", "lat": 60.463376, "lon": 22.109561, "darkness": "It's dark", "acceleration": 247.3, "speed": 12.8, "surface": "Slippery", "traffic": "Low", "friction": 0.102},
    13: {"date": "2026-01-09", "time": "03:21", "town": "Paimio", "street": "Valtatie", "lat": 60.421144, "lon": 22.728092, "darkness": "It's dark", "acceleration": 134.8, "speed": 18.0, "surface": "Slippery", "traffic": "Low", "friction": 0.106},
    14: {"date": "2026-01-09", "time": "03:41", "town": "Salo", "street": "Meriniityntie", "lat": 60.391843, "lon": 23.082998, "darkness": "It's dark", "acceleration": 48.2, "speed": 51.4, "surface": "Slippery", "traffic": "Low", "friction": 0.127},
    15: {"date": "2026-01-09", "time": "03:44", "town": "Salon suuralue", "street": "Joensuunkatu", "lat": 60.384240, "lon": 23.100431, "darkness": "It's dark", "acceleration": 108.6, "speed": 91.4, "surface": "Slippery", "traffic": "Low", "friction": 0.127},
    16: {"date": "2026-01-09", "time": "16:27", "town": "Halikko", "street": "Valtatie", "lat": 60.404377, "lon": 22.820764, "darkness": "It's not dark", "acceleration": 152.2, "speed": 77.0, "surface": "Slippery", "traffic": "High", "friction": 0.160},
    17: {"date": "2026-01-09", "time": "16:32", "town": "Paimio", "street": "Valtatie", "lat": 60.421258, "lon": 22.728743, "darkness": "It's not dark", "acceleration": 152.2, "speed": 40.1, "surface": "Slippery", "traffic": "High", "friction": 0.100},
    18: {"date": "2026-01-09", "time": "22:29", "town": "Salo", "street": "Järvenkyläntie", "lat": 60.323705, "lon": 22.939734, "darkness": "It's dark", "acceleration": 52.9, "speed": 13.1, "surface": "Slippery", "traffic": "Low", "friction": 0.107},
}

# helper function to extract press number from user text
def get_press_from_text(text):
    match = re.search(r'press\s*(\d+)', text.lower())
    if match:
        press_num = int(match.group(1))
        if press_num in press_data:
            return press_num
    return 1

# helper function to classify grip safely in Python
def classify_grip(friction):
    if friction < 0.2:
        return "dangerous"
    elif friction <= 0.5:
        return "moderately dangerous"
    else:
        return "safe"

# A function that can be called elsewhere to provide an analysis
def predict_analysis(text):
    # detect press number
    press_num = get_press_from_text(text)
    event = press_data[press_num]

    # get weekday name from date
    day_name = datetime.strptime(event["date"], "%Y-%m-%d").strftime("%A")

    # classify grip in Python first
    grip_class = classify_grip(event["friction"])

    # Our system prompt in this example
    system_prompt = "Your task is to analyze the potential reason why the driver of a truck marked a dangerous situation on the road. " \
    "Road grip (friction) has already been classified using these thresholds: below 0.2 = dangerous, 0.2 to 0.5 = moderately dangerous, above 0.5 = safe. " \
    f"For this event, the road grip classification is: {grip_class}. " \
    "Do not confuse road grip classification with overall situation risk. A road grip value above 0.5 is safe in terms of grip, but the overall situation may still be risky because of speed, acceleration, darkness, traffic, or road surface conditions. " \
    "The road grip classification provided in the prompt is final and must not be changed. " \
    "If a sensor value appears unusually high, mention that it may indicate a sudden shock, abrupt maneuver, or a possible sensor spike, but do not ignore it. " \
    "There are sensors in the truck, and their current values were: " \
    f"Press: {press_num}. " \
    f"Truck acceleration: {event['acceleration']} m/s². " \
    f"Truck speed: {event['speed']} km/h. " \
    f"Date: {event['date']} ({day_name}). " \
    f"Time: {event['time']}. " \
    f"City and street: {event['town']} (Finland), {event['street']}. " \
    f"Latitude: {event['lat']}. " \
    f"Longitude: {event['lon']}. " \
    f"Darkness level: {event['darkness']}. " \
    f"Road surface condition: {event['surface']}. " \
    f"Traffic: {event['traffic']}. " \
    f"Road grip / friction: {event['friction']} ({grip_class}). " \
    "Use the press number if the user asks which press it was. " \
    "First state the date and weekday when relevant. Then state the road grip classification. Then explain the most likely reason for the dangerous situation using acceleration, speed, darkness, road surface condition, and traffic. " \
    "Only use this information if the user specifically asks about the reason for marking down the dangerous situation. " \
    "Be concise in your answers."

    # initialize our message data
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text}
    ]

    # Build prompt
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=500)

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    # build response
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    print(response)

# wrapper for gradio chat
def chat_response(message, history):
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        predict_analysis(message)

    result = buffer.getvalue().strip()

    if not result:
        result = "No response generated."

    return result

demo = gr.ChatInterface(
    fn=chat_response,
    title="Truck Dangerous Situation Analyzer",
    description="Ask about a press event, for example: 'What happened at press 7?'",
    examples=[
        "What happened at press 1?",
        "Explain the dangerous situation for press 7",
        "What was the likely reason for press 12?",
        "Explain the most likely cause of the dangerous situation using all provided sensor data for press 8. Include the date and weekday."
        "Explain the most likely cause of the dangerous situation using ALL provided sensor data for press 8. Specifically consider acceleration, speed, road condition, and darkness. Be concise but include reasoning."
    ]
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [4]:
predict_analysis("Analyze press 8 using all provided sensor data. First state the date. Then evaluate road grip safety. Finally, explain the most likely cause of the dangerous situation, considering acceleration, speed, road condition, and darkness. Be concise but include reasoning.")

Date: 2025-11-27 (Thursday)
Road Grip Classification: Safe

The truck's press value of 8 indicates a high level of pressure on the road surface. However, the road grip classification is safe, suggesting that the grip is sufficient to handle the truck's speed and acceleration.

The most likely cause of the dangerous situation is the high acceleration of 337.5 m/s². This rapid acceleration may have caused the truck to lose traction on the dry road surface, leading to a potentially hazardous situation. The speed of 0.9 km/h is also a contributing factor, as it may have increased the likelihood of losing control. The lack of darkness and dry road surface condition further reduce the risk of accidents, but the combination of high acceleration and speed may still pose a risk.
